# 基本設定

In [3]:
# 可調參數
EXCEL_PATH = "/Users/wanghao/8k/8K_Filings.xlsx"
SHEET_INDEX = 0

HTML_DIR = "/Volumes/One Touch/8k_file_html"
TXT_DIR  = "/Volumes/One Touch/8k_file_txt"
OTHER_DIR = "/Volumes/One Touch/8k_file_neither"

DRY_RUN = True   # True: 先跑少量測試
DRY_RUN_N = 50

MIN_INTERVAL = 0.25
USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome Safari (Contact: haowang5277@gmail.com)"
)
SEC_PREFIX = "https://www.sec.gov/"


# 匯入套件

In [5]:
import re
import os
import time
from typing import List, Optional

import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, parse_qs


# 讀 Excel 與去重複函式

In [7]:
start = time.perf_counter()

KEEP = ["coname", "cik", "accession"]

def keep_and_dedupe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [c.strip() for c in out.columns]
    return out[KEEP].drop_duplicates(subset=["cik", "accession"], keep="first")

df_raw = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_INDEX)
res = keep_and_dedupe(df_raw)

print(f"刪減後（列, 欄）= {res.shape}")
print(res.head())

end = time.perf_counter()
print(f"花費時間：{end - start:.2f} 秒")


刪減後（列, 欄）= (835233, 3)
                               coname     cik             accession
0                   SOUTHERN UNION CO  203248  0000913907-94-000002
1  CLEVELAND ELECTRIC ILLUMINATING CO   20947  0000774197-94-000001
2                    TOLEDO EDISON CO  352049  0000774197-94-000001
3                    BROOKE GROUP LTD   59440  0000950123-94-000010
4                    MDC HOLDINGS INC  773141  0000950109-94-000014
花費時間：37.94 秒


# HTTP 與速率控制

In [9]:
class FatalError(RuntimeError):
    pass

_last_call_ts = 0.0

def _rate_limit(): #控制下載頻率，確保請求間隔不低於 MIN_INTERVAL 秒
    global _last_call_ts
    now = time.monotonic()
    wait = (_last_call_ts + MIN_INTERVAL) - now
    if wait > 0:
        time.sleep(wait)
    _last_call_ts = time.monotonic()

def _get(url: str, stream: bool = False) -> requests.Response: #對單一 URL 發送 GET 請求，內建重試、錯誤處理和限速
    _rate_limit()
    try:
        r = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=30, stream=stream)
        if r.status_code == 429:
            ra = r.headers.get("Retry-After")
            sleep_s = int(ra) if ra and ra.isdigit() else 5
            print(f"[429] Too Many Requests: 等待 {sleep_s} 秒後重試...")
            time.sleep(sleep_s)
            _rate_limit()
            r = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=30, stream=stream)
        if r.status_code != 200:
            raise FatalError(f"HTTP {r.status_code}: {url}")
        return r
    except requests.RequestException as e:
        raise FatalError(f"請求失敗: {url}: {e}")


# 找到要的下載連結函式

In [11]:
_HTML_EXTS = (".htm", ".html")
_TXT_EXTS = (".txt",)

def _find_primary_doc_url(index_html: str) -> str:
    soup = BeautifulSoup(index_html, "html.parser")

    # 找 Document Format Files 表格
    table = None
    for t in soup.find_all("table"):
        ths = [th.get_text(strip=True).lower() for th in t.find_all("th")]
        if {"seq", "description", "document", "type", "size"} <= set(ths):
            table = t
            break
    if table is None:
        raise FatalError("找不到 Document Format Files 表格")

    ths = [th.get_text(strip=True).lower() for th in table.find_all("th")]
    col = {name: ths.index(name) for name in ["seq", "description", "document", "type", "size"]}

    # 目標表單：8-K 家族
    keywords = ("8-k", "8-k/a")

    candidates_html = []
    candidates_txt = []

    for tr in table.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) <= max(col.values()):
            continue

        type_txt = tds[col["type"]].get_text(strip=True).lower()

        # 等於或前綴（更準確，避免誤抓）
        if not any(type_txt == k or type_txt.startswith(k) for k in keywords):
            continue

        for a in tds[col["document"]].find_all("a", href=True):
            raw = a["href"].strip()
            low = raw.lower()

            # 處理 iXBRL 連結：/ix?doc=...
            if low.startswith("/ix?") or "/ix?" in low:
                q = urlparse(raw).query
                doc = parse_qs(q).get("doc", [None])[0]
                if doc:
                    raw = doc
                    low = raw.lower()

            # 避免 viewer 殼連結
            if "viewer" in low and ("doc=" not in low) and not low.endswith(_HTML_EXTS + _TXT_EXTS):
                continue

            # 只接受 .htm/.html/.txt
            if not low.endswith(_HTML_EXTS + _TXT_EXTS):
                continue

            abs_url = urljoin(SEC_PREFIX, raw)
            if low.endswith(_HTML_EXTS):
                candidates_html.append(abs_url)
            else:
                candidates_txt.append(abs_url)

    # 優先回傳 HTML，再回傳 TXT
    if candidates_html:
        return candidates_html[0]
    if candidates_txt:
        return candidates_txt[0]

    # --- Fallback：沒匹配到 Type，挑第一個 HTML/TXT ---
    for tr in table.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) <= max(col.values()):
            continue
        for a in tds[col["document"]].find_all("a", href=True):
            raw = a["href"].strip()
            low = raw.lower()

            if low.startswith("/ix?") or "/ix?" in low:
                q = urlparse(raw).query
                doc = parse_qs(q).get("doc", [None])[0]
                if doc:
                    raw = doc
                    low = raw.lower()

            if low.endswith(_HTML_EXTS):
                return urljoin(SEC_PREFIX, raw)
            if low.endswith(_TXT_EXTS):
                candidates_txt.append(urljoin(SEC_PREFIX, raw))

    if candidates_txt:
        return candidates_txt[0]

    raise FatalError("找不到符合 8-K/8-KA 的主檔連結，且沒有可用的 fallback")


# 組出 Index URL + 下載函式

In [13]:
BATCH_SIZE = 5000
HTML_DIR = "/Volumes/One Touch/8k_file_html"
TXT_DIR  = "/Volumes/One Touch/8k_file_txt"
OTHER_DIR = "/Volumes/One Touch/8k_file_other"

_HTML_EXTS = (".htm", ".html")
_TXT_EXTS = (".txt",)

def _sanitize_filename(name: str) -> str:
    name = re.sub(r'[\\/*?:"<>|]', "", str(name))
    name = name.replace(" ", "_")
    return name.strip()

def _count_existing_files(root: str, exts: tuple[str, ...]) -> int:
    total = 0
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if fn.lower().endswith(exts):
                total += 1
    return total

def _batch_dir(base_dir: str, batch_id: int) -> str:
    d = os.path.join(base_dir, f"batch_{batch_id:03d}")
    os.makedirs(d, exist_ok=True)
    return d

def _make_index_url(cik: str, accession: str) -> str:
    acc_no_dash = accession.replace("-", "")
    return f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_no_dash}/{accession}-index.html"

def download_8k_batched(df: pd.DataFrame, limit: int) -> List[str]:
    os.makedirs(HTML_DIR, exist_ok=True)
    os.makedirs(TXT_DIR, exist_ok=True)
    # os.makedirs(OTHER_DIR, exist_ok=True)

    # 1) 計算目前已存在的檔案數，用來續跑
    existing_html = _count_existing_files(HTML_DIR, _HTML_EXTS)
    existing_txt  = _count_existing_files(TXT_DIR, _TXT_EXTS)
    # 如果你也存 OTHER，就把 OTHER 的數也算進來
    existing_total = existing_html + existing_txt  # + existing_other

    total = min(limit, len(df))
    saved, errors = [], []
    start_t = time.time()

    for i, (_, row) in enumerate(df.iloc[:total].iterrows(), 1):
        try:
            coname_safe = _sanitize_filename(row.coname)
            accession_safe = _sanitize_filename(row.accession)
            url = _make_index_url(row.cik, row.accession)

            # 2) 下載主檔 URL（這裡預設抓 8-K）
            index_resp = _get(url)
            primary_url = _find_primary_doc_url(
                index_resp.text  # 你的 _find_primary_doc_url 已經鎖 8-K / 8-K/A
            )
            resp = _get(primary_url, stream=True)

            # 3) 依「全域序號 = 既有檔數 + 本批第 i 筆 - 1」決定 batch
            global_idx = existing_total + (i - 1)
            batch_id = global_idx // BATCH_SIZE

            # 4) 依副檔名決定落點目錄（在 HTML_DIR / TXT_DIR 各自的 batch_XXX）
            filename = f"{coname_safe}_{accession_safe}"
            low = primary_url.lower()
            if low.endswith(_HTML_EXTS):
                out_dir = _batch_dir(HTML_DIR, batch_id)
                out_path = os.path.join(out_dir, f"{filename}.html")
            elif low.endswith(_TXT_EXTS):
                out_dir = _batch_dir(TXT_DIR, batch_id)
                out_path = os.path.join(out_dir, f"{filename}.txt")
            else:
                # 如果你要分 OTHER，就改到 OTHER_DIR
                out_dir = _batch_dir(OTHER_DIR, batch_id)  # 或 OTHER_DIR
                out_path = os.path.join(out_dir, f"{filename}.dat")

            # 5) 寫檔
            with open(out_path, "wb") as f:
                for chunk in resp.iter_content(chunk_size=1024 * 64):
                    if chunk:
                        f.write(chunk)

            saved.append(out_path)
            print(f"[OK] {i}/{total} -> {out_path}")

        except Exception as e:
            errors.append({
                "coname": getattr(row, "coname", ""),
                "accession": getattr(row, "accession", ""),
                "index_url": url,
                "error": str(e)
            })
            print(f"[ERR] {i}/{total}: {e}")

    print(f"\n✅ 成功下載 {len(saved)} 檔，失敗 {len(errors)} 檔；耗時 {time.time() - start_t:.2f} 秒")
    if errors:
        pd.DataFrame(errors).to_csv("download_errors.csv", index=False, encoding="utf-8-sig")
        print(f"❗ 已記錄失敗清單到 download_errors.csv ({len(errors)} 筆)")
    return saved


# 嘗試下載

In [43]:
if DRY_RUN:
    df_slice = res.iloc[:DRY_RUN_N]
else:
    df_slice = res

print(f"本次處理筆數：{len(df_slice)}")
saved_files = download_8k_batched(df_slice, limit=len(df_slice))


本次處理筆數：50
[OK] 1/50 -> /Volumes/One Touch/8k_file_txt/batch_000/SOUTHERN_UNION_CO_0000913907-94-000002.txt
[OK] 2/50 -> /Volumes/One Touch/8k_file_txt/batch_000/CLEVELAND_ELECTRIC_ILLUMINATING_CO_0000774197-94-000001.txt
[OK] 3/50 -> /Volumes/One Touch/8k_file_txt/batch_000/TOLEDO_EDISON_CO_0000774197-94-000001.txt
[OK] 4/50 -> /Volumes/One Touch/8k_file_txt/batch_000/BROOKE_GROUP_LTD_0000950123-94-000010.txt
[OK] 5/50 -> /Volumes/One Touch/8k_file_txt/batch_000/MDC_HOLDINGS_INC_0000950109-94-000014.txt
[OK] 6/50 -> /Volumes/One Touch/8k_file_txt/batch_000/CENTRAL_HUDSON_GAS_&_ELECTRIC_CORP_0000018647-94-000001.txt
[OK] 7/50 -> /Volumes/One Touch/8k_file_txt/batch_000/CITIZENS_UTILITIES_CO_0000020520-94-000001.txt
[OK] 8/50 -> /Volumes/One Touch/8k_file_txt/batch_000/PRICE_COMMUNICATIONS_CORP_0000897446-94-000004.txt
[OK] 9/50 -> /Volumes/One Touch/8k_file_txt/batch_000/KAYDON_CORP_0000740694-94-000001.txt
[OK] 10/50 -> /Volumes/One Touch/8k_file_txt/batch_000/PENTAIR_INC_0000077360-94

# 實際分批下載

In [63]:
# 手動輸入要處理的起點與終點（包含起點，不包含終點）
start = int(input("請輸入起始索引 (例如 0)："))
end = int(input("請輸入結束索引 (例如 50000)："))

# 檢查輸入範圍是否合法
if start < 0 or end > len(res) or start >= end:
    raise ValueError(f"❌ 輸入範圍不正確！總筆數為 {len(res)}，start 必須 < end 且都在範圍內。")

# 擷取該範圍的資料
df_slice = res.iloc[start:end]

print(f"處理第 {start} ~ {end} 筆（共 {len(df_slice)} 筆）")
saved_files = download_8k_batched(df_slice, limit=len(df_slice))

print("下載完成！")


請輸入起始索引 (例如 0)： 824000
請輸入結束索引 (例如 50000)： 835233


處理第 824000 ~ 835233 筆（共 11233 筆）
[OK] 1/11233 -> /Volumes/One Touch/8k_file_html/batch_163/HELIOS_TECHNOLOGIES,_INC._0000950170-25-047827.html
[OK] 2/11233 -> /Volumes/One Touch/8k_file_html/batch_163/SEMPRA_0001032208-25-000020.html
[OK] 3/11233 -> /Volumes/One Touch/8k_file_html/batch_163/BXP,_Inc._0001656423-25-000014.html
[OK] 4/11233 -> /Volumes/One Touch/8k_file_html/batch_163/YUM_BRANDS_INC_0001193125-25-067752.html
[OK] 5/11233 -> /Volumes/One Touch/8k_file_html/batch_163/BOSTON_PROPERTIES_LTD_PARTNERSHIP_0001656423-25-000014.html
[OK] 6/11233 -> /Volumes/One Touch/8k_file_html/batch_163/MICROSTRATEGY_Inc_0000950170-25-047219.html
[OK] 7/11233 -> /Volumes/One Touch/8k_file_html/batch_163/GUARANTY_BANCSHARES_INC_TX_0000950170-25-047785.html
[OK] 8/11233 -> /Volumes/One Touch/8k_file_html/batch_163/CF_BANKSHARES_INC._0001070680-25-000012.html
[OK] 9/11233 -> /Volumes/One Touch/8k_file_html/batch_163/CANNAPHARMARX,_INC._0001654954-25-003608.html
[OK] 10/11233 -> /Volumes/One Touch

In [ ]:
到824000

In [55]:
x = 690000 + 56317
print(x)

746317


In [61]:
800000+24000

824000